In [1]:
import pandas as pd
import numpy as np
import os
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, f1_score
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "../../data/split/essays"
EXP_DIR  = "../../experiment/SVM"
MODEL_DIR = "../../model/SVM"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(EXP_DIR, exist_ok=True)

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val_df   = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

print(f"Train: {train_df.shape[0]} samples")
print(f"Val:   {val_df.shape[0]} samples")
print(f"Test:  {test_df.shape[0]} samples")
print(f"Columns: {train_df.columns.tolist()}")
print(f"Trait distribution (train):")
traits = ["cEXT", "cNEU", "cAGR", "cCON", "cOPN"]
for t in traits:
    print(f"  {t}: {dict(train_df[t].value_counts())}")

Train: 1974 samples
Val:   247 samples
Test:  247 samples
Columns: ['text', 'cEXT', 'cNEU', 'cAGR', 'cCON', 'cOPN', 'label']
Trait distribution (train):
  cEXT: {'high': 1022, 'low': 952}
  cNEU: {'low': 988, 'high': 986}
  cAGR: {'high': 1046, 'low': 928}
  cCON: {'high': 1004, 'low': 970}
  cOPN: {'high': 1017, 'low': 957}


In [2]:
# TF-IDF vectorizer: normalized TF-IDF vectors as input features
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),    # unigrams + bigrams for richer features
    min_df=2,              # ignore very rare terms
    sublinear_tf=True      # apply sublinear TF scaling (1 + log(tf))
)

X_train = vectorizer.fit_transform(train_df["text"])
X_val   = vectorizer.transform(val_df["text"])
X_test  = vectorizer.transform(test_df["text"])

print(f"TF-IDF vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"X_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"X_test shape:  {X_test.shape}")

TF-IDF vocabulary size: 10000
X_train shape: (1974, 10000)
X_val shape:   (247, 10000)
X_test shape:  (247, 10000)


In [3]:
import joblib
import os
import time
from sklearn.metrics import f1_score

TRAITS = ["cOPN", "cCON", "cEXT", "cAGR",  "cNEU"]
TRAIT_NAMES = {
    "cOPN": "Openness",
    "cCON": "Conscientiousness",
    "cEXT": "Extraversion",
    "cAGR": "Agreeableness",
    "cNEU": "Neuroticism",
}

run_id = time.strftime('%Y%m%d-%H%M%S')

results = {}

for trait in TRAITS:
    sep = "=" * 60
    print(f"\n{sep}")
    print(f"Training SVM for {trait} ({TRAIT_NAMES[trait]})")
    print(f"{sep}")

    y_train = train_df[trait]
    y_val   = val_df[trait]
    y_test  = test_df[trait]

    clf = SVC(kernel="rbf", C=10.0, gamma=0.1, random_state=42)
    clf.fit(X_train, y_train)

    pred_train = clf.predict(X_train)
    pred_val   = clf.predict(X_val)
    pred_test  = clf.predict(X_test)

    train_acc = accuracy_score(y_train, pred_train)
    val_acc   = accuracy_score(y_val,   pred_val)
    test_acc  = accuracy_score(y_test,  pred_test)
    test_f1   = f1_score(y_test, pred_test, average="weighted", zero_division=0)

    print(f"  Train Accuracy: {train_acc:.4f}")
    print(f"  Val Accuracy:   {val_acc:.4f}")
    print(f"  Test Accuracy:  {test_acc:.4f}")
    print(f"  Test F1 (weighted): {test_f1:.4f}")

    # high first, then low
    report_test  = classification_report(y_test, pred_test,  labels=["high", "low"], digits=4, zero_division=0)
    report_val   = classification_report(y_val,  pred_val,   labels=["high", "low"], digits=4, zero_division=0)
    report_train = classification_report(y_train, pred_train, labels=["high", "low"], digits=4, zero_division=0)

    results[trait] = {
        "model": clf,
        "train_acc": train_acc,
        "val_acc":   val_acc,
        "test_acc":  test_acc,
        "test_f1":   test_f1,
        "report_test":  report_test,
        "report_val":   report_val,
        "report_train": report_train,
        "pred_test": pred_test,
        "y_test": y_test
    }

    # Save model and vectorizer for this trait
    trait_model_path = os.path.join(MODEL_DIR, f"svm_{trait}.joblib")
    joblib.dump(clf, trait_model_path)
    print(f"  Saved model: {trait_model_path}")

    print(f"\nClassification Report (Test):")
    print(report_test)

# Save the shared TF-IDF vectorizer (same for all traits)
vectorizer_path = os.path.join(MODEL_DIR, "tfidf_vectorizer.joblib")
joblib.dump(vectorizer, vectorizer_path)
print(f"\nSaved TF-IDF vectorizer: {vectorizer_path}")

# Save mapping metadata
meta = {
    "traits": TRAITS,
    "trait_names": TRAIT_NAMES,
    "vectorizer_path": vectorizer_path,
    "model_dir": MODEL_DIR,
    "run_id": run_id,
}
meta_path = os.path.join(MODEL_DIR, "metadata.joblib")
joblib.dump(meta, meta_path)
print(f"Saved metadata: {meta_path}")


Training SVM for cOPN (Openness)
  Train Accuracy: 0.9959
  Val Accuracy:   0.5911
  Test Accuracy:  0.5466
  Test F1 (weighted): 0.5464
  Saved model: ../../model/SVM\svm_cOPN.joblib

Classification Report (Test):
              precision    recall  f1-score   support

        high     0.5581    0.5669    0.5625       127
         low     0.5339    0.5250    0.5294       120

    accuracy                         0.5466       247
   macro avg     0.5460    0.5460    0.5460       247
weighted avg     0.5464    0.5466    0.5464       247


Training SVM for cCON (Conscientiousness)
  Train Accuracy: 0.9944
  Val Accuracy:   0.5547
  Test Accuracy:  0.5101
  Test F1 (weighted): 0.5096
  Saved model: ../../model/SVM\svm_cCON.joblib

Classification Report (Test):
              precision    recall  f1-score   support

        high     0.5152    0.5440    0.5292       125
         low     0.5043    0.4754    0.4895       122

    accuracy                         0.5101       247
   macro avg  

In [4]:
# # Load saved models and run inference on new text

# loaded_vectorizer = joblib.load(os.path.join(MODEL_DIR, "tfidf_vectorizer.joblib"))
# loaded_meta       = joblib.load(os.path.join(MODEL_DIR, "metadata.joblib"))

# print("Loaded metadata:", loaded_meta)
# print(f"\nAvailable traits: {loaded_meta['traits']}")

# # Example: predict on the test set using loaded models
# print("\n" + "=" * 70)
# print("Inference with Loaded Models on Test Set")
# print("=" * 70)

# for trait in TRAITS:
#     loaded_clf = joblib.load(os.path.join(MODEL_DIR, f"svm_{trait}.joblib"))

#     # Transform test text with loaded vectorizer
#     X_test_loaded = loaded_vectorizer.transform(test_df["text"])
#     pred_loaded = loaded_clf.predict(X_test_loaded)

#     acc = accuracy_score(test_df[trait], pred_loaded)
#     print(f"{trait} ({TRAIT_NAMES[trait]}): Test Accuracy = {acc:.4f}")

# # Predict on a custom essay
# custom_essays = [
#     "I love meeting new people and going to parties. I am always full of energy and enjoy being the center of attention.",
#     "I prefer staying alone and reading books. I get anxious in large crowds and need a lot of quiet time.",
# ]

# print("\n" + "=" * 70)
# print("Custom Essay Predictions")
# print("=" * 70)

# X_custom = loaded_vectorizer.transform(custom_essays)

# for i, essay in enumerate(custom_essays):
#     print(f"\nEssay {i+1}: {essay[:80]}...")
#     for trait in TRAITS:
#         loaded_clf = joblib.load(os.path.join(MODEL_DIR, f"svm_{trait}.joblib"))
#         pred = loaded_clf.predict(X_custom[i])[0]
#         print(f"  {trait} ({TRAIT_NAMES[trait]}): {pred}")


In [5]:
# Print per-trait summary to stdout
print(f"\n{'='*70}")
print(f"SUMMARY: SVM Classification Results across OCEAN Traits")
print(f"{'='*70}")
print(f"{'Trait':<8} {'Test Acc':>10} {'Test F1':>10}")
print(f"{'-'*70}")

summary_data = []
for trait in TRAITS:
    r = results[trait]
    print(f"{trait:<8} {r['test_acc']:>10.4f} {r['test_f1']:>10.4f}")
    summary_data.append((trait, r['test_acc'], r['test_f1']))

print(f"{'-'*70}")
avg_test_acc = np.mean([s[1] for s in summary_data])
avg_test_f1  = np.mean([s[2] for s in summary_data])
print(f"{'AVERAGE':<8} {avg_test_acc:>10.4f} {avg_test_f1:>10.4f}")
print(f"{'='*70}")


SUMMARY: SVM Classification Results across OCEAN Traits
Trait      Test Acc    Test F1
----------------------------------------------------------------------
cOPN         0.5466     0.5464
cCON         0.5101     0.5096
cEXT         0.5587     0.5557
cAGR         0.4818     0.4793
cNEU         0.5304     0.5300
----------------------------------------------------------------------
AVERAGE      0.5255     0.5242


In [6]:
# Compute combined classification report for all traits
os.makedirs(EXP_DIR, exist_ok=True)

sep = "=" * 70
report_lines = [
    sep,
    "Personality Trait Detection — Classification Report",
    sep,
    f"Run ID      : {run_id}",
    f"Model       : SVM (TF-IDF + RBF kernel, C=10, gamma=0.1)",
    f"Test file   : test.csv",
    f"Test samples: {len(test_df)}",
    sep,
]

for trait in TRAITS:
    gt   = results[trait]["y_test"]
    pred = results[trait]["pred_test"]
    acc  = results[trait]["test_acc"]
    f1   = results[trait]["test_f1"]

    report_str = classification_report(
        gt, pred,
        labels=["high", "low"],
        target_names=["high", "low"],
        digits=4,
        zero_division=0,
    )

    report_lines.extend([
        f"\n--- {trait} ({TRAIT_NAMES[trait]}) ---",
        f"Accuracy: {acc:.4f} | F1 (weighted): {f1:.4f}",
        report_str,
    ])

report_lines.append(sep)

report_path = os.path.join(EXP_DIR, "classification_report.txt")
with open(report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))
print(f"Saved: {report_path}")

Saved: ../../experiment/SVM\classification_report.txt


In [7]:
# Save raw predictions with stable filename
test_pred_df = test_df.copy()
for trait in TRAITS:
    test_pred_df[f"{trait}_pred"] = results[trait]["pred_test"]
raw_pred_path = os.path.join(EXP_DIR, "raw_predictions.csv")
test_pred_df.to_csv(raw_pred_path, index=False)
print(f"Raw predictions saved: {raw_pred_path}")

Raw predictions saved: ../../experiment/SVM\raw_predictions.csv
